# Практична робота 2. Ускладнення параметричної моделі L-кронштейна

## Мета
Розширити параметричну модель технічної деталі:
- додати ребра жорсткості;
- зробити кількість монтажних отворів змінною;
- забезпечити автоматичне рівномірне та симетричне розташування отворів;
- сформувати спрощене параметричне креслення з основними розмірами.

**Важливе розмежування:** це навчальна параметрична геометрична модель на `numpy`/`matplotlib`. Вона демонструє форму, розташування елементів, проєкції та розмірні залежності, але не виконує булеве вирізання отворів у твердотільному CAD-ядрі та не є виробничим кресленням.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Circle
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import ipywidgets as widgets
from IPython.display import display

plt.rcParams['figure.figsize'] = (15, 10)


## 1. Параметри та геометричні залежності

Габарити:
- `W` — ширина кронштейна;
- `H` — висота вертикальної полиці;
- `D` — глибина горизонтальної полиці;
- `T` — товщина полиць.

Монтаж:
- `n` — кількість отворів у кожному ряду;
- `d` — діаметр отворів;
- `E` — відступ центрів крайніх отворів від бокових країв.

Для `n` отворів координати по ширині визначаються рівномірно:

\[
z_i=E+i\frac{W-2E}{n-1},\quad i=0,1,\ldots,n-1
\]

Для `n=1` центр є на осі симетрії: `z=W/2`.

Ребра жорсткості:
- `G` — довжина катета трикутного ребра в площині XY;
- `Rt` — товщина ребра вздовж ширини Z;
- два ребра розташовуються біля бокових країв кронштейна.

In [ ]:
def hole_positions(W, E, n):
    if n == 1:
        return np.array([W/2])
    return np.linspace(E, W-E, n)


def validate(W,H,D,T,n,d,E,G,Rt):
    if min(W,H,D,T,d,E,G,Rt) <= 0:
        raise ValueError("Усі розміри мають бути додатними.")
    if T >= min(H,D):
        raise ValueError("T має бути меншою за H і D.")
    if n < 1:
        raise ValueError("Кількість отворів має бути не меншою за 1.")
    if n > 1 and 2*E >= W:
        raise ValueError("Відступи E не залишають місця для ряду отворів.")
    if E <= d/2:
        raise ValueError("Відступ E має перевищувати радіус отвору.")
    if G <= T or G >= min(H,D)-T:
        raise ValueError("Оберіть G більше T, але менше за доступні габарити.")
    if Rt <= 0 or Rt >= W:
        raise ValueError("Перевірте товщину ребра Rt.")
    if G + d/2 >= min(H-T,D-T):
        raise ValueError("Ребро або отвори виходять за доступну область полиць.")


## 2. Допоміжні функції побудови 3D-геометрії

Основний корпус — L-подібний профіль, екструдований уздовж Z. Ребра жорсткості — трикутні пластини біля двох бокових країв. Контури отворів показані на поверхнях умовно.

In [ ]:
def add_extruded_polygon(ax, profile, z0, thickness, color="#9bbbd0", alpha=0.92):
    a = [(x,y,z0) for x,y in profile]
    b = [(x,y,z0+thickness) for x,y in profile]
    faces = [a,b]
    for i in range(len(profile)):
        j=(i+1)%len(profile)
        faces.append([a[i],a[j],b[j],b[i]])
    ax.add_collection3d(Poly3DCollection(
        faces, facecolors=color, edgecolors="#263746",
        linewidths=0.8, alpha=alpha
    ))


def add_hole_rings(ax, W, T, H, D, d, zvals, xhole, yhole):
    a=np.linspace(0,2*np.pi,100)
    r=d/2
    for z in zvals:
        # На горизонтальній полиці — коло в площині XZ
        ax.plot(xhole+r*np.cos(a), np.full_like(a,T), z+r*np.sin(a),
                color="darkred",lw=1.8)
        # На вертикальній полиці — коло в площині YZ
        ax.plot(np.full_like(a,T), yhole+r*np.cos(a), z+r*np.sin(a),
                color="darkred",lw=1.8)


def draw_dimension(ax, p1, p2, offset, label, text_offset=0):
    # Розмірна лінія для 2D-креслення; p1,p2 — точки у координатах осей
    x1,y1=p1; x2,y2=p2
    ox,oy=offset
    ax.annotate("", xy=(x2+ox,y2+oy), xytext=(x1+ox,y1+oy),
                arrowprops=dict(arrowstyle="<->",lw=1,color="black"))
    ax.text((x1+x2)/2+ox, (y1+y2)/2+oy+text_offset, label,
            ha="center", va="bottom", fontsize=9)


## 3. Побудова моделі, проєкцій і креслення

На кресленні показано габаритні розміри та основні параметри отворів. Розмірні лінії формуються з тих самих параметрів, що й геометрія.

In [ ]:
def build_model(W=120,H=90,D=80,T=6,n=3,d=10,E=18,G=28,Rt=5):
    validate(W,H,D,T,n,d,E,G,Rt)
    zvals=hole_positions(W,E,n)
    xhole=D*0.70
    yhole=H*0.68
    profile=[(0,0),(D,0),(D,T),(T,T),(T,H),(0,H)]

    fig=plt.figure(figsize=(16,11))
    ax=fig.add_subplot(221,projection="3d")
    top=fig.add_subplot(222)
    front=fig.add_subplot(223)
    side=fig.add_subplot(224)

    # Основний корпус
    add_extruded_polygon(ax,profile,0,W)

    # Два трикутні ребра жорсткості: переріз у XY, екструзія вздовж Z
    gusset=[(T,T),(T+G,T),(T,T+G)]
    z_g1=max(0, E-Rt/2)
    z_g2=min(W-Rt, W-E-Rt/2)
    add_extruded_polygon(ax,gusset,z_g1,Rt,color="#d5a45b",alpha=0.98)
    if abs(z_g2-z_g1)>1e-8:
        add_extruded_polygon(ax,gusset,z_g2,Rt,color="#d5a45b",alpha=0.98)

    # Отвори — умовні контури
    add_hole_rings(ax,W,T,H,D,d,zvals,xhole,yhole)

    ax.set(xlim=(0,D*1.08),ylim=(0,H*1.08),zlim=(0,W),
           xlabel="X — глибина",ylabel="Y — висота",zlabel="Z — ширина",
           title="3D-модель: корпус, ребра та отвори")
    ax.set_box_aspect((D,H,W))
    ax.view_init(elev=24,azim=-55)

    # Вид зверху: контур горизонтальної полиці; ребра показані умовно проєкцією
    top.fill([0,D,D,0],[0,0,W,W],color="#a9c4d6",alpha=.75)
    for z in zvals:
        top.add_patch(Circle((xhole,z),d/2,fill=False,color="darkred",lw=1.8))
    top.set(xlim=(-12,D+12),ylim=(-12,W+12),xlabel="X — глибина",
            ylabel="Z — ширина",title="Вигляд зверху")
    top.set_aspect("equal")
    top.grid(True)

    # Вид спереду: вертикальна полиця
    front.fill([0,W,W,0],[0,0,H,H],color="#a9c4d6",alpha=.75)
    for z in zvals:
        front.add_patch(Circle((z,yhole),d/2,fill=False,color="darkred",lw=1.8))
    front.set(xlim=(-12,W+12),ylim=(-12,H+12),xlabel="Z — ширина",
              ylabel="Y — висота",title="Вигляд спереду")
    front.set_aspect("equal")
    front.grid(True)

    # Вид збоку: L-профіль і трикутні ребра як проєкція
    closed=profile+[profile[0]]
    side.fill([p[0] for p in closed],[p[1] for p in closed],
              color="#a9c4d6",alpha=.75)
    side.plot([p[0] for p in closed],[p[1] for p in closed],color="#263746")
    tri=gusset+[gusset[0]]
    side.fill([p[0] for p in tri],[p[1] for p in tri],
              color="#d5a45b",alpha=.95)
    side.set(xlim=(-12,D+12),ylim=(-12,H+12),xlabel="X — глибина",
             ylabel="Y — висота",title="Вигляд збоку")
    side.set_aspect("equal")
    side.grid(True)

    fig.suptitle(f"L-кронштейн | W={W}; H={H}; D={D}; T={T}; "
                 f"n={n}; d={d}; E={E}; G={G}; Rt={Rt} мм",fontsize=13)
    plt.tight_layout()
    plt.show()

build_model()


## 4. Інтерактивна зміна параметрів

Перевірте, як змінюються ребра та кількість отворів. Зверніть увагу: відстані між центрами отворів обчислюються автоматично.

In [ ]:
Ww=widgets.IntSlider(value=120,min=80,max=180,step=5,description="W, мм")
Hh=widgets.IntSlider(value=90,min=60,max=150,step=5,description="H, мм")
Dd=widgets.IntSlider(value=80,min=60,max=140,step=5,description="D, мм")
Tt=widgets.IntSlider(value=6,min=3,max=12,step=1,description="T, мм")
nn=widgets.IntSlider(value=3,min=1,max=6,step=1,description="К-сть n")
hd=widgets.IntSlider(value=10,min=6,max=16,step=1,description="d, мм")
Ee=widgets.IntSlider(value=18,min=12,max=30,step=1,description="E, мм")
Gg=widgets.IntSlider(value=28,min=15,max=45,step=1,description="G, мм")
Rr=widgets.IntSlider(value=5,min=3,max=10,step=1,description="Rt, мм")

controls=widgets.VBox([
    widgets.HBox([Ww,Hh,Dd]),
    widgets.HBox([Tt,nn,hd]),
    widgets.HBox([Ee,Gg,Rr])
])
out=widgets.interactive_output(build_model,{
    "W":Ww,"H":Hh,"D":Dd,"T":Tt,"n":nn,"d":hd,"E":Ee,"G":Gg,"Rt":Rr
})
display(controls,out)


## 5. Автоматичне формування розмірного ескізу

Це окрема спрощена схема. Вона не замінює оформлене за стандартами робоче креслення, але демонструє принцип: розміри беруться безпосередньо з параметрів моделі.

In [ ]:
def dimensioned_sketch(W=120,H=90,D=80,T=6,n=3,d=10,E=18):
    zvals=hole_positions(W,E,n)
    fig,axs=plt.subplots(1,2,figsize=(14,5))

    # Фронтальний вигляд
    ax=axs[0]
    ax.add_patch(plt.Rectangle((0,0),W,H,fill=False,lw=2))
    for z in zvals:
        ax.add_patch(Circle((z,H*0.68),d/2,fill=False,color="darkred",lw=1.5))
    ax.set_xlim(-18,W+18); ax.set_ylim(-20,H+22)
    ax.set_aspect("equal"); ax.grid(True)
    ax.set_title("Фронтальний ескіз")
    ax.set_xlabel("Ширина, мм"); ax.set_ylabel("Висота, мм")
    draw_dimension(ax,(0,0),(W,0),(0,-12),f"W = {W}")
    draw_dimension(ax,(0,0),(0,H),(-12,0),f"H = {H}",text_offset=0)

    # Бічний вигляд
    ax=axs[1]
    prof=[(0,0),(D,0),(D,T),(T,T),(T,H),(0,H),(0,0)]
    ax.fill([p[0] for p in prof],[p[1] for p in prof],
            facecolor="#a9c4d6",alpha=.5)
    ax.plot([p[0] for p in prof],[p[1] for p in prof],lw=2,color="#263746")
    ax.add_patch(Polygon([(T,T),(T+min(28,D-T-2),T),
                         (T,T+min(28,H-T-2))],
                        closed=True,facecolor="#d5a45b",alpha=.8))
    ax.set_xlim(-18,D+20); ax.set_ylim(-20,H+22)
    ax.set_aspect("equal"); ax.grid(True)
    ax.set_title("Бічний профіль")
    ax.set_xlabel("Глибина, мм"); ax.set_ylabel("Висота, мм")
    draw_dimension(ax,(0,0),(D,0),(0,-12),f"D = {D}")
    draw_dimension(ax,(0,0),(T,0),(0,-5),f"T = {T}",text_offset=-3)

    fig.suptitle(f"Параметричний ескіз | отвори: n={n}, d={d}, крайній відступ E={E} мм")
    plt.tight_layout()
    plt.show()

dimensioned_sketch()


## 6. Самостійні завдання

1. Змініть кількість отворів `n` від 2 до 5. Запишіть формулу відстані між сусідніми отворами.
2. Збільште ширину `W`, не змінюючи `E`. Чи зберігається симетрія?
3. Збільшіть катет ребра `G`. Які геометричні обмеження потрібно врахувати?
4. Змініть товщину ребра `Rt`. Чи впливає вона на габарити кронштейна?
5. Запропонуйте правило, за яким кількість отворів могла б автоматично залежати від ширини деталі.

## 7. Контрольні запитання

1. Які параметри задаються користувачем безпосередньо?
2. Які координати отворів є похідними?
3. Як формула розміщення отворів забезпечує симетрію?
4. Яку функцію виконують ребра жорсткості?
5. Чим параметричний ескіз відрізняється від повного виробничого креслення?
6. Які перевірки потрібні, щоб параметри не створювали геометрично неможливу деталь?

**Очікуваний результат:** студент може змінювати кількість отворів і геометрію ребер, пояснювати залежності та отримувати оновлені проєкції з параметрів моделі.